# ControlNet — End-to-End Walkthrough

From data acquisition to a controlled generation, this notebook tells the whole story. It is a **thin narrative layer**: every heavy step calls the project's library code (`src/`) or scripts — nothing is reimplemented here.

1. Problem & approach (zero convolutions)
2. Data acquisition (COCO + captions)
3. Canny conditioning
4. Architecture & correctness (parity test)
5. Training (loss curve / sudden convergence)
6. Results (condition → generation)
7. Evaluation (FID / CLIP / Canny fidelity)

In [ ]:
import sys; sys.path.append('..')  # repo root, so `import src...` works from notebooks/
import json, numpy as np, torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
ROOT = Path('..').resolve()

## 1. Problem & approach
ControlNet adds spatial control (here: Canny edges) to a **frozen** Stable Diffusion U-Net. It clones the encoder into a trainable copy and connects it back through **zero convolutions** — 1×1 convs initialised to zero, so training begins as an exact no-op and the pretrained model is never harmed. See `src/models/zero_conv.py`.

## 2. Data acquisition
COCO `val2017` + the official captions, prepared into JSONL by `scripts/prepare_coco.py` (run once from the repo root: `python scripts/prepare_coco.py --root data/coco`).

In [ ]:
from src.data.dataset import CocoCannyDataset
ds = CocoCannyDataset(str(ROOT / 'data/coco'), split='val', image_size=512)
print(len(ds), 'val samples; example caption:', ds.samples[0]['caption'])

## 3. Canny conditioning
The condition is generated on the fly from each image. Top row = source image, bottom = Canny edges fed to ControlNet.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    s = ds[i]
    img = ((s['pixel_values'] + 1) / 2).permute(1, 2, 0).numpy()
    cond = s['conditioning_pixel_values'].permute(1, 2, 0).numpy()
    axes[0, i].imshow(img); axes[0, i].axis('off')
    axes[1, i].imshow(cond); axes[1, i].axis('off')
plt.tight_layout(); plt.show()

## 4. Architecture & correctness
Build the hand-written ControlNet from the SD U-Net and count trainable vs frozen params. Its correctness is proven by `tests/test_parity_with_diffusers.py`, which asserts its residuals match `diffusers.ControlNetModel` to ~1e-4 (`pytest tests/`).

In [ ]:
# Optional (downloads SD v1.5): build the model and report parameter counts.
# from diffusers import UNet2DConditionModel
# from src.models.controlnet import ControlNet
# unet = UNet2DConditionModel.from_pretrained('runwayml/stable-diffusion-v1-5', subfolder='unet')
# cn = ControlNet.from_unet(unet)
# print('trainable:', sum(p.numel() for p in cn.parameters()))

## 5. Training
Training runs from the repo root via `accelerate launch scripts/train.py --config configs/<cfg>.yaml` (locally with `smoke_local.yaml`, on RunPod with `runpod_24gb.yaml` — see `docs/runpod.md`). Plot the logged loss to show the **sudden convergence** ControlNet is known for.

In [ ]:
# Plot loss from a wandb export or a local log, once a run exists.
# losses = json.load(open(ROOT / 'outputs/loss_log.json'))
# plt.plot(losses); plt.xlabel('step'); plt.ylabel('loss'); plt.show()

## 6. Results
Load the trained checkpoint and show **condition → generation** for a few validation conditions.

In [ ]:
# CKPT = ROOT / 'outputs/runpod_24gb/controlnet-final.safetensors'
# from src.inference.generate import ControlNetInference
# infer = ControlNetInference(controlnet_path=str(CKPT), condition_type='canny')
# s = ds[0]; cond = (s['conditioning_pixel_values'] * 255).byte().permute(1,2,0).numpy()
# out = infer.generate(s['caption'], Image.fromarray(cond), seed=0)
# fig, ax = plt.subplots(1, 3, figsize=(12, 4))
# ax[0].imshow(cond); ax[1].imshow(out); ax[2].imshow(((s['pixel_values']+1)/2).permute(1,2,0))
# for a in ax: a.axis('off')
# plt.show()

## 7. Evaluation
`python scripts/evaluate.py --controlnet <ckpt> --config configs/runpod_24gb.yaml` writes `outputs/eval_report.json` with FID, CLIP score, and Canny fidelity (edge-F1 + SSIM).

In [ ]:
# report = json.load(open(ROOT / 'outputs/eval_report.json')); print(json.dumps(report, indent=2))